# `rfc3161_client` smoke test

Trail of Bits' RFC 3161 Time-Stamp Protocol client (Rust/PyO3 core). Everything
below runs offline: requests are built and round-tripped locally, error contracts
are probed through public APIs, and the verifier is built from a locally generated
test certificate. No real time-stamping authority (TSA) is contacted.

## Example 1: Build an RFC 3161 timestamp request

In [ ]:
# Example 1: Build an RFC 3161 timestamp request
from rfc3161_client import TimestampRequestBuilder
from rfc3161_client.tsp import TimeStampRequest

request = TimestampRequestBuilder().data(b"important artifact bytes").build()
assert isinstance(request, TimeStampRequest)
assert request.version == 1
assert request.cert_req is True  # default: ask the TSA to include its certificate
assert request.policy is None  # no policy OID requested by default
assert isinstance(request.nonce, int)  # fresh random nonce by default
assert len(request.message_imprint.message) == 64  # default hash: SHA-512
print(f"version={request.version} nonce={request.nonce}")
print("OK — default request built entirely offline")

## Example 2: The message imprint holds the hash of the data

In [ ]:
# Example 2: The request carries the hash of the data, never the data itself
import hashlib

from rfc3161_client import HashAlgorithm, TimestampRequestBuilder

payload = b"release tarball\x00\x01\x02"
request = (
    TimestampRequestBuilder()
    .data(payload)
    .hash_algorithm(HashAlgorithm.SHA256)
    .build()
)
imprint = request.message_imprint
assert imprint.message == hashlib.sha256(payload).digest()
assert imprint.hash_algorithm.dotted_string == "2.16.840.1.101.3.4.2.1"  # id-sha256
assert payload not in request.as_bytes()  # only the digest is embedded
print("OK — imprint == sha256(data); OID", imprint.hash_algorithm.dotted_string)

## Example 3: DER round trip of a request

In [ ]:
# Example 3: DER round trip: serialize with as_bytes(), parse again, compare fields
# parse_timestamp_request is the package's own Rust binding; the package has no public
# wrapper for decoding a request.
from rfc3161_client import HashAlgorithm, TimestampRequestBuilder
from rfc3161_client._rust import parse_timestamp_request

original = TimestampRequestBuilder().data(b"round-trip me").hash_algorithm(HashAlgorithm.SHA512).build()
der = original.as_bytes()
assert der[0] == 0x30  # ASN.1 SEQUENCE
reparsed = parse_timestamp_request(der)
assert reparsed.nonce == original.nonce
assert reparsed.message_imprint.message == original.message_imprint.message
assert reparsed.as_bytes() == der

# Without a nonce the encoding is fully deterministic:
a = TimestampRequestBuilder().nonce(nonce=False).data(b"x").build().as_bytes()
b = TimestampRequestBuilder().nonce(nonce=False).data(b"x").build().as_bytes()
assert a == b
print(f"OK — {len(der)}-byte DER round-trips; nonce-free encoding is deterministic")

## Example 4: Error contracts of the builder and the response decoder

In [ ]:
# Example 4: Error contracts of the request builder and the response decoder
# A client must reject a malformed TSA response with ValueError, not crash.
from rfc3161_client import TimestampRequestBuilder, decode_timestamp_response

try:
    TimestampRequestBuilder().data(b"")
    raise AssertionError("empty data must be rejected")
except ValueError as error:
    print("empty data ->", error)
try:
    TimestampRequestBuilder().data(b"a").data(b"b")
    raise AssertionError("data may only be set once")
except ValueError as error:
    print("double data ->", error)
try:
    TimestampRequestBuilder().hash_algorithm("md5")
    raise AssertionError("non-HashAlgorithm must be rejected")
except TypeError as error:
    print("bad hash ->", error)
try:
    decode_timestamp_response(b"garbage, not DER")
    raise AssertionError("garbage must not decode")
except ValueError as error:
    print("bad response ->", error)
print("OK — all documented error contracts hold")

## Example 5: Verifier built from a generated test certificate

In [ ]:
# Example 5: Build a Verifier from a locally generated test certificate
import datetime

from cryptography import x509
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.x509.oid import NameOID
from rfc3161_client import VerifierBuilder
from rfc3161_client.verify import Verifier

key = ec.generate_private_key(ec.SECP256R1())
name = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, "Test TSA")])
now = datetime.datetime.now(datetime.timezone.utc)
cert = (
    x509.CertificateBuilder().subject_name(name).issuer_name(name)
    .public_key(key.public_key()).serial_number(x509.random_serial_number())
    .not_valid_before(now).not_valid_after(now + datetime.timedelta(days=1))
    .sign(key, hashes.SHA256())
)
verifier = (VerifierBuilder().add_root_certificate(cert).tsa_certificate(cert)
            .common_name("Test TSA").nonce(1234).build())
assert isinstance(verifier, Verifier) and callable(verifier.verify_message)

# A verifier without any root certificate is refused.
try:
    VerifierBuilder().build()
except ValueError as error:
    print("no-roots guard ->", error)
else:
    raise AssertionError("VerifierBuilder().build() without a root certificate should raise ValueError")
print("OK — offline Verifier built from a generated test certificate")